# GP5 — nonlinear-regime (LAOS) training, Table 5

Identifies the coefficients $c_0 \dots c_7$ of eq. (7),

$$\dot\gamma = c_0 + c_1\sigma + c_2\dot\sigma + c_3\sigma^2 + c_4\sigma\dot\sigma
+ c_5\dot\sigma\sigma^2 + c_6\sigma^3 + c_7\dot\sigma\sigma^3 ,$$

from the stress-controlled LAOS data of GP5 ($\sigma_0$ = 100 Pa, $\omega$ = 1 rad/s).
Each sample age is trained independently: a RhINN maps $(t, \sigma_0) \mapsto \gamma$,
automatic differentiation gives $\dot\gamma$, and STRidge selects the active terms inside
the ADO loop (Section 2.2).

Two term libraries were used. The **full** library keeps all eight terms of eq. (7). Because
the full fits became unstable in start-up shear, every age was re-fitted with a **reduced**
library without the constant and $\sigma^3$ terms, and Table 5 reports the reduced fit. The
notebook of the original full-library run was not recovered; its settings are assumed to
be those of the reduced run.

All training settings are in `rhinn_sindy/settings.py`.

* **Output:** `results/retrained/GP5_LAOS_full.csv` and `GP5_LAOS_reduced.csv`, written
  after every age. Table 5 itself (`results/coefficients/GP5_LAOS.csv`) is never
  overwritten.
* **Runtime:** about 40 min per age on a laptop CPU.
* **Hyperparameters:** the settings in `rhinn_sindy/settings.py` are the ones used for the
  paper. Moving from one sample age to another changes the stress and strain magnitudes by
  orders of magnitude, so an age (or a new material) may need some hyperparameter tuning,
  mainly the loss weights, learning rate, number of rounds and STRidge tolerance.

Even with the original settings, retrained coefficients differ from run to run (random
initialisation and collocation points) and can differ noticeably from Table 5.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from rhinn_sindy import figures
from rhinn_sindy.reproduce import available_ages, compare_with_paper, run_laos

MATERIAL = "GP5"
tf.random.set_seed(1234)
np.random.seed(1234)

## Choose the ages

Ages available for training (as labelled in Table 5) and the library the paper
reports at each:

In [ ]:
available_ages(MATERIAL, "LAOS")

In [ ]:
# "all" for the whole of Table 5, or a list of ages in minutes, e.g. [6, 76].
AGES = "all"
# "paper" uses the library in the table above; "full" or "reduced" forces one.
LIBRARY = "paper"

## Training data

In [ ]:
figures.laos_training_data(MATERIAL)
plt.show()

## Train

In [ ]:
results = run_laos(MATERIAL, AGES, LIBRARY)

## Compare with the fits behind Table 5

In [ ]:
compare_with_paper(MATERIAL, "LAOS")

## Check: integrate eq. (7) with the trained coefficients

The stress used in training (nominal or fitted sinusoid) drives eq. (7) from the first
measured strain (up to four of the trained ages).

In [ ]:
figures.laos_fit(MATERIAL, results)
plt.show()